# Extraction / chunking / ingestion MSO

## Finalite du notebook

Ce notebook est la reference pour traiter les documents internes MSO situes dans `data/in/MSO`.
Il couvre tout le cycle de preparation des donnees, depuis la lecture des fichiers sources jusqu'a l'insertion optionnelle en base vectorielle.

Il sait :

- lire des `pdf`, `pptx` et `docx`
- extraire un texte exploitable meme quand le PDF est degrade
- detecter automatiquement le type de document
- reconstruire des sections metier plutot que faire un simple chunking aveugle
- produire les artefacts `documents`, `sections` et `chunks`
- calculer les embeddings locaux et Scaleway
- inserer le resultat dans `rag_documents`, `rag_sections` et `rag_chunks_mso`

La strategie centrale est la suivante :

1. conserver au maximum la structure reelle du document
2. transformer chaque section en unite RAG exploitable
3. construire une pseudo-question derivee du titre de section pour rester proche des requetes utilisateurs

Autrement dit, on ne veut ni perdre la hierarchie documentaire, ni produire des chunks trop bruts pour le retrieval.

## Ce qu'il faut comprendre pour presenter le notebook

### Entrees et sorties

- Entree principale : `data/in/MSO/**`
- Textes extraits : `data/out/MSO/**`
- Chunks JSONL : `data/out/chunked/*.jsonl`
- Embeddings : `data/out/*.jsonl`, `*.parquet`, `*.npy`
- Base cible par defaut : `rag_chunks_mso`

### Les 4 grands types de documents detectes

- `guide` : document avec titres et sous-titres classiques
- `faq` : document de type questions/reponses numerotees
- `process` : logigramme, mode operatoire, suite d'etapes
- `table_matrix` : document tabulaire ou matriciel ou la mise en page est determinante

### Logique de parsing

- un `guide` est sectionne par niveaux de titres
- une `faq` est reconstruite en blocs question/reponse
- un `process` devient une suite d'etapes avec branche et acteur si possible
- une `table_matrix` privilegie `pdftotext -layout` pour ne pas casser le tableau

### Pourquoi ce notebook existe

Les notebooks plus generiques ne suffisent pas pour certains sous-dossiers MSO car les documents melangent plusieurs realites : PDF scannes, tableaux, guides tres structures, FAQ et supports bureautiques. Ce notebook est donc devenu la couche de reference pour absorber cette heterogeneite sans casser les cas deja couverts.

### Ordre d'execution du notebook

1. charger l'environnement et les chemins
2. definir les fonctions d'extraction (`pdf`, `pptx`, `docx`, OCR)
3. definir les heuristiques de parsing
4. exporter les textes sources vers `data/out/MSO`
5. construire les datasets `documents / sections / chunks`
6. calculer les embeddings
7. faire l'upsert SQL seulement si `MSO_UPSERT_TO_DB=1`

### Quand l'utiliser

- pour tester un nouveau sous-dossier dans `data/in/MSO`
- pour regenerer proprement les JSONL d'un lot MSO
- pour inserer un lot valide dans Scalingo, Scaleway staging ou Scaleway prod

### Ce qu'il faut verifier a chaque run

- que les `.txt` extraits sont lisibles
- que le mode detecte pour chaque document est coherent
- que les counts `documents / sections / chunks` sont plausibles
- que les embeddings ne sont pas vides
- que la base cible correspond bien a l'environnement voulu

### Variables d'environnement a connaitre absolument

- `MSO_INPUT_PATTERNS` : quels fichiers traiter
- `MSO_TXT_PATTERNS` : quels `.txt` reutiliser pour le chunking
- `MSO_CHUNKS_JSONL`, `MSO_DOCS_JSONL`, `MSO_SECTIONS_JSONL` : sorties structurees
- `MSO_EMB_OUT_JSONL` : chunks avec embeddings
- `MSO_DB_TARGET` : `scalingo`, `scaleway_staging`, `scaleway_prod`, `custom`
- `MSO_UPSERT_TO_DB` : active l'insertion SQL
- `MSO_REPLACE_EXISTING_DOCS` : remplace les documents deja presents
- `MSO_OCR_FALLBACK` : active l'OCR de secours

### Message de passation simple

Si quelqu'un reprend ce notebook, la bonne approche est : cibler un sous-dossier, lancer un dry run sans upsert, verifier les artefacts locaux, puis seulement activer l'insertion en base.

## Etape 1 - Bootstrap du run

Les deux cellules suivantes font le minimum pour preparer l'execution :

- charger le fichier `.env` si present
- repositionner le notebook a la racine du repo

Pourquoi c'est important : tout le notebook depend de chemins relatifs comme `data/in/MSO` et `data/out/MSO`. Si cette etape est sautee ou si le notebook est execute depuis le mauvais repertoire, les fichiers ne seront pas trouves ou seront ecrits au mauvais endroit.

In [1]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

In [2]:
%cd ..

/Users/omar.gueddari/work/assistant-rh


## Etape 2 - Configuration globale et dependances

Cette cellule centralise toute la configuration du notebook : imports, variables d'environnement, chemins d'entree/sortie, choix de la base cible, parametres OCR, parametres d'embedding et resolution du DSN.

C'est la cellule la plus importante pour piloter un run. En pratique, c'est ici qu'on change :

- le sous-dossier a traiter via `MSO_INPUT_PATTERNS`
- les artefacts de sortie JSONL
- la base cible via `MSO_DB_TARGET`
- le comportement d'upsert et de remplacement
- les seuils OCR si un PDF est mal extrait

Le `print(...)` final sert de check rapide avant de poursuivre : il permet de verifier que le notebook pointe bien vers les bons chemins, la bonne base et le bon modele.

In [3]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable
from glob import glob
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import subprocess
import tempfile
import time
import unicodedata
import uuid

import numpy as np
import pandas as pd
import psycopg
import requests
import torch
from psycopg.rows import dict_row
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

BASE_IN = Path(os.getenv("MSO_BASE_IN", "./data/in/MSO"))
BASE_OUT = Path(os.getenv("MSO_BASE_OUT", "./data/out/MSO"))
INPUT_FILES = [p.strip() for p in os.getenv("MSO_INPUT_FILES", "").split(",") if p.strip()]
INPUT_PATTERNS = [p.strip() for p in os.getenv("MSO_INPUT_PATTERNS", "./data/in/MSO/**/*.pdf,./data/in/MSO/**/*.pptx,./data/in/MSO/**/*.docx").split(",") if p.strip()]
TXT_GLOB = [p.strip() for p in os.getenv("MSO_TXT_PATTERNS", "./data/out/MSO/**/*.txt").split(",") if p.strip()]
OUT_JSONL = os.getenv("MSO_CHUNKS_JSONL", "./data/out/chunked/mso_chunks_qna.jsonl")
OUT_DOCS_JSONL = os.getenv("MSO_DOCS_JSONL", "./data/out/chunked/mso_documents.jsonl")
OUT_SECTIONS_JSONL = os.getenv("MSO_SECTIONS_JSONL", "./data/out/chunked/mso_sections.jsonl")
OUT_JSONL_WITH_EMB = os.getenv("MSO_EMB_OUT_JSONL", "./data/out/mso_chunks_baai_bge_m3_with_emb.jsonl")
OUT_PARQUET = os.getenv("MSO_EMB_OUT_PARQUET", "./data/out/mso_chunks_baai_bge_m3.parquet")
OUT_NPY = os.getenv("MSO_EMB_OUT_NPY", "./data/out/mso_chunks_baai_bge_m3.npy")
TABLE = os.getenv("MSO_TABLE", "rag_chunks_mso")
SCHEMA = os.getenv("PGSCHEMA", "public")
MODEL_NAME = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")
EMBED_COL = os.getenv("EMBEDDING_COLUMN", "embedding_m3")
BATCH_SIZE = int(os.getenv("MSO_EMBED_BATCH_SIZE", "64"))
NORMALIZE = True
GENERATE_BGE_SCW = os.getenv("MSO_GENERATE_BGE_SCW", "1").strip().lower() in {"1", "true", "yes", "y", "on"}
SCALEWAY_BGE_MODEL = os.getenv("SCALEWAY_BGE_MODEL", "bge-multilingual-gemma2")
SCALEWAY_BGE_BATCH_SIZE = int(os.getenv("MSO_BGE_BATCH_SIZE", "16"))
SCALEWAY_BGE_TIMEOUT = int(os.getenv("MSO_BGE_TIMEOUT", "30"))
SCALEWAY_BASE_URL = (os.getenv("SCALEWAY_BASE_URL", "https://api.scaleway.ai/11aa88cb-ec5b-4df9-bcb4-e9e82576ae58/v1") or "").rstrip("/")
SCALEWAY_API_KEY = os.getenv("SCALEWAY_API_KEY", "").strip()
OCR_FALLBACK_ENABLED = os.getenv("MSO_OCR_FALLBACK", "1").strip().lower() in {"1", "true", "yes", "y", "on"}
OCR_LANG = os.getenv("MSO_OCR_LANG", "fra")
OCR_DPI = int(os.getenv("MSO_OCR_DPI", "200"))
OCR_MIN_CHARS = int(os.getenv("MSO_OCR_MIN_CHARS", "80"))
OCR_MIN_ALPHA_CHARS = int(os.getenv("MSO_OCR_MIN_ALPHA_CHARS", "30"))

MSO_DB_TARGET = os.getenv("MSO_DB_TARGET", "scalingo").strip().lower()
SCALINGO_URL = os.getenv("SCALINGO_URL", "")
SCW_POSTGRES_DSN = os.getenv("SCW_POSTGRES_DSN", "")
SCW_POSTGRES_DSN_STAGING = os.getenv("SCW_POSTGRES_DSN_STAGING", "")
SCW_POSTGRES_DSN_PROD = os.getenv("SCW_POSTGRES_DSN_PROD", SCW_POSTGRES_DSN)

def make_scalingo_tunnel_url(scalingo_url: str, host: str = "127.0.0.1", port: int = 10001) -> str:
    u = urlparse(scalingo_url.replace("postgres://", "postgresql://"))
    q = dict(parse_qsl(u.query))
    q["sslmode"] = "disable"
    return urlunparse(
        u._replace(
            netloc=f"{u.username}:{u.password}@{host}:{port}",
            query=urlencode(q),
        )
    )

def resolve_database_url() -> str:
    explicit = (os.getenv("DATABASE_URL", "") or "").strip()
    if explicit:
        return explicit
    if MSO_DB_TARGET == "scalingo":
        return make_scalingo_tunnel_url(SCALINGO_URL) if SCALINGO_URL else ""
    if MSO_DB_TARGET in {"scaleway", "scaleway_prod", "scw", "prod"}:
        return SCW_POSTGRES_DSN_PROD or SCW_POSTGRES_DSN
    if MSO_DB_TARGET in {"scaleway_staging", "staging", "scw_staging"}:
        return SCW_POSTGRES_DSN_STAGING
    if MSO_DB_TARGET in {"custom", "local"}:
        return explicit
    raise ValueError(f"MSO_DB_TARGET non supporte: {MSO_DB_TARGET}")

DATABASE_URL = resolve_database_url()
DB_CFG = dict(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    dbname=os.getenv("PGDATABASE", "postgres"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", ""),
    sslmode=os.getenv("PGSSLMODE", "require"),
)

def pg_conn():
    if DATABASE_URL:
        return psycopg.connect(DATABASE_URL, row_factory=dict_row)
    return psycopg.connect(**DB_CFG, row_factory=dict_row)

MSO_NAMESPACE = uuid.UUID("c5cdb7de-f8c4-4f8e-9b3d-c9ff7e7d4b72")

def sha1_u(s: str) -> str:
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def sha256_text(s: str) -> str:
    return hashlib.sha256((s or "").encode("utf-8")).hexdigest()

def stable_uuid_from_parts(namespace: uuid.UUID, *parts: object) -> str:
    key = ":".join(str(part) for part in parts)
    return str(uuid.uuid5(namespace, key))

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def normalize_text(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in s.split("\n")]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def expand_inputs(files: list[str], patterns: list[str]) -> list[str]:
    ordered: list[str] = []
    seen: set[str] = set()
    for f in files:
        if f and f not in seen:
            seen.add(f)
            ordered.append(f)
    for pat in patterns:
        for p in sorted(glob(pat, recursive=True)):
            if p not in seen:
                seen.add(p)
                ordered.append(p)
    return ordered

def load_dotenv_key_candidates(env_path: Path, key_name: str) -> list[str]:
    if not env_path.exists():
        return []
    values: list[str] = []
    for line in env_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        key, value = stripped.split("=", 1)
        if key.strip() != key_name:
            continue
        candidate = value.strip().strip('"').strip("'")
        if candidate and candidate not in values:
            values.append(candidate)
    return values

def compute_thematique(path: str) -> str:
    parts = [pp for pp in Path(path).parts if pp not in (".", "")]
    lower = [pp.lower() for pp in parts]
    if "data" in lower:
        tail = parts[lower.index("data") + 1 :]
    else:
        tail = parts[:]
    if tail and tail[0].lower() in ("in", "out"):
        tail = tail[1:]
    if tail and "." in tail[-1][1:]:
        tail = tail[:-1]
    return "/".join(tail).strip("/")

def slugify_short_id(value: str, prefix: str = "MSO") -> str:
    raw = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_") or prefix
    short_id = f"{prefix}_{raw}"[:48].rstrip("_")
    suffix = sha1_u(value)[:12]
    return f"{short_id}_{suffix}"[:64]

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4) if text else 0

def normalize_vector(vec: list[float]) -> list[float]:
    arr = np.asarray(vec, dtype=np.float32)
    norm = float(np.linalg.norm(arr))
    return (arr / norm).tolist() if norm > 0 else arr.tolist()

def scaleway_embed_text(text: str, *, retries: int = 5) -> list[float]:
    if not SCALEWAY_API_KEY:
        raise RuntimeError("SCALEWAY_API_KEY manquante pour generer embedding_bge_scw.")
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.post(
                f"{SCALEWAY_BASE_URL}/embeddings",
                headers={
                    "Authorization": f"Bearer {SCALEWAY_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={"model": SCALEWAY_BGE_MODEL, "input": text},
                timeout=SCALEWAY_BGE_TIMEOUT,
            )
            if response.status_code == 429:
                time.sleep(min(30, 2 ** attempt))
                continue
            response.raise_for_status()
            payload = response.json()
            return normalize_vector(payload["data"][0]["embedding"])
        except Exception as exc:
            last_error = exc
            time.sleep(min(30, 2 ** attempt))
    if last_error is not None:
        raise last_error
    raise RuntimeError("Echec embedding_bge_scw sans erreur explicite.")

def is_valid_scaleway_api_key(api_key: str) -> bool:
    try:
        response = requests.post(
            f"{SCALEWAY_BASE_URL}/embeddings",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={"model": SCALEWAY_BGE_MODEL, "input": "test"},
            timeout=SCALEWAY_BGE_TIMEOUT,
        )
        response.raise_for_status()
        payload = response.json()
        return bool(payload.get("data"))
    except Exception:
        return False

def resolve_scaleway_api_key(env_path: Path = Path('.env')) -> str:
    candidates: list[str] = []
    if SCALEWAY_API_KEY:
        candidates.append(SCALEWAY_API_KEY)
    for candidate in load_dotenv_key_candidates(env_path, "SCALEWAY_API_KEY"):
        if candidate not in candidates:
            candidates.append(candidate)
    if not candidates:
        return ""
    for candidate in candidates:
        if is_valid_scaleway_api_key(candidate):
            return candidate
    raise RuntimeError("Aucune SCALEWAY_API_KEY candidate ne permet d'appeler l'API embeddings Scaleway.")

SCALEWAY_API_KEY = resolve_scaleway_api_key()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print({
    "BASE_IN": str(BASE_IN),
    "BASE_OUT": str(BASE_OUT),
    "OUT_JSONL": OUT_JSONL,
    "OUT_JSONL_WITH_EMB": OUT_JSONL_WITH_EMB,
    "TABLE": TABLE,
    "MSO_DB_TARGET": MSO_DB_TARGET,
    "DATABASE_URL_SET": bool(DATABASE_URL),
    "MODEL_NAME": MODEL_NAME,
    "GENERATE_BGE_SCW": GENERATE_BGE_SCW,
    "SCALEWAY_BGE_MODEL": SCALEWAY_BGE_MODEL,
    "SCALEWAY_API_KEY_SET": bool(SCALEWAY_API_KEY),
    "OCR_FALLBACK_ENABLED": OCR_FALLBACK_ENABLED,
    "OCR_LANG": OCR_LANG,
    "OCR_DPI": OCR_DPI,
    "device": device,
})

/Users/omar.gueddari/.pyenv/versions/3.12.6/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'BASE_IN': 'data/in/MSO', 'BASE_OUT': 'data/out/MSO', 'OUT_JSONL': './data/out/chunked/mso_remuneration_revalorisation_chunks_qna.jsonl', 'OUT_JSONL_WITH_EMB': './data/out/mso_remuneration_revalorisation_chunks_with_emb.jsonl', 'TABLE': 'rag_chunks_mso', 'MSO_DB_TARGET': 'scalingo', 'DATABASE_URL_SET': True, 'MODEL_NAME': 'BAAI/bge-m3', 'GENERATE_BGE_SCW': True, 'SCALEWAY_BGE_MODEL': 'bge-multilingual-gemma2', 'SCALEWAY_API_KEY_SET': True, 'OCR_FALLBACK_ENABLED': True, 'OCR_LANG': 'fra', 'OCR_DPI': 200, 'device': 'cpu'}


## Etape 3 - Extraction des fichiers sources vers des `.txt`

Cette etape contient les fonctions de lecture bas niveau pour `pdf`, `pptx` et `docx`, puis exporte chaque document source dans `data/out/MSO/**`.

Logique cle :

- PDF natif : extraction texte standard
- PDF tabulaire : preference pour `pdftotext -layout`
- PDF degrade : fallback OCR avec `pdftoppm` + `tesseract`
- PPTX : reconstruction slide par slide
- DOCX : lecture XML directe du document

Le DataFrame affiche en sortie sert de tableau de controle : on y voit le fichier source, le type, le `.txt` produit, le nombre de caracteres, les doublons eventuels et le statut d'export.

In [4]:
def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def looks_like_table_matrix_text(text: str) -> bool:
    norm = normalize_text(text).lower()
    if "type d'actes" not in norm:
        return False
    entity_hit = "entité de gestion" in norm or "entite de gestion" in norm
    degree_hits = len(re.findall(r"\d+°", norm))
    entity_values = sum(norm.count(label) for label in ("déconcentré", "deconcentre", "drh-bpeco", "cbcm"))
    section_hits = sum(norm.count(label) for label in ("modalités de service", "modalites de service", "absences et congés", "absences et conges", "disciplinaire", "propositions financières", "propositions financieres"))
    return entity_hit and ((degree_hits >= 5 and entity_values >= 5) or (entity_values >= 8 and section_hits >= 2))

def read_pdf_layout_text(path: str) -> str:
    try:
        cp = subprocess.run(["pdftotext", "-layout", path, "-"], check=True, capture_output=True, text=True)
        return cp.stdout or ""
    except Exception:
        return ""

def is_text_extraction_too_poor(text: str) -> bool:
    content = normalize_text(re.sub(r"\[PAGE\s+\d+\]", " ", text or ""))
    alpha_chars = len(re.findall(r"[A-Za-zÀ-ÿ]", content))
    return len(content) < OCR_MIN_CHARS or alpha_chars < OCR_MIN_ALPHA_CHARS

def ocr_pdf_to_text(path: str) -> str:
    if not OCR_FALLBACK_ENABLED:
        return ""
    try:
        with tempfile.TemporaryDirectory(prefix="mso_ocr_") as tmpdir:
            prefix = str(Path(tmpdir) / "page")
            subprocess.run(["pdftoppm", "-r", str(OCR_DPI), "-png", path, prefix], check=True, capture_output=True, text=True)
            image_paths = sorted(Path(tmpdir).glob("page-*.png"))
            parts: list[str] = []
            for i, image_path in enumerate(image_paths, start=1):
                cp = subprocess.run(["tesseract", str(image_path), "stdout", "-l", OCR_LANG, "--psm", "6"], check=True, capture_output=True, text=True)
                txt = normalize_text(cp.stdout or "")
                if txt:
                    parts.append(f"[PAGE {i}]\n{txt}")
            return normalize_text("\n\n".join(parts))
    except FileNotFoundError:
        return ""
    except Exception as exc:
        print({"ocr_fallback_error": path, "error": str(exc)})
        return ""

def read_pdf_to_text(path: str) -> str:
    layout_text = read_pdf_layout_text(path)
    if layout_text and looks_like_table_matrix_text(layout_text):
        return normalize_text(layout_text)
    reader = PdfReader(path)
    parts: list[str] = []
    for i, page in enumerate(reader.pages, start=1):
        txt = normalize_text(page.extract_text() or "")
        if txt:
            parts.append(f"[PAGE {i}]\n{txt}")
    pdf_text = normalize_text("\n\n".join(parts))
    if not is_text_extraction_too_poor(pdf_text):
        return pdf_text
    layout_norm = normalize_text(layout_text)
    if layout_norm and not is_text_extraction_too_poor(layout_norm):
        return layout_norm
    ocr_text = ocr_pdf_to_text(path)
    if ocr_text:
        print({"ocr_fallback_used": path, "chars": len(ocr_text)})
        return ocr_text
    return pdf_text or layout_norm

def read_pptx_to_text(path: str) -> str:
    from pptx import Presentation

    prs = Presentation(path)
    parts: list[str] = []
    for slide_index, slide in enumerate(prs.slides, start=1):
        texts: list[str] = []
        for shape in slide.shapes:
            text = getattr(shape, "text", "") or ""
            text = normalize_text(text)
            if text:
                texts.append(text)
        if not texts:
            continue
        title = texts[0].split("\n", 1)[0].strip()
        body = []
        for block in texts[1:]:
            for line in block.split("\n"):
                line = line.strip()
                if line:
                    body.append(f"- {line}")
        section_lines = [f"{slide_index}. {title}"]
        if body:
            section_lines.extend(body)
        parts.append(f"[SLIDE {slide_index}]\n" + "\n".join(section_lines))
    return normalize_text("\n\n".join(parts))

def read_docx_to_text(path: str) -> str:
    from zipfile import ZipFile
    import xml.etree.ElementTree as ET

    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    with ZipFile(path) as zf:
        root = ET.fromstring(zf.read("word/document.xml"))
    parts: list[str] = []
    for para in root.findall(".//w:p", ns):
        texts = [node.text or "" for node in para.findall(".//w:t", ns)]
        text = normalize_text("".join(texts))
        if text:
            parts.append(text)
    return normalize_text("\n".join(parts))

def read_document_to_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return read_pdf_to_text(str(path))
    if suffix == ".pptx":
        return read_pptx_to_text(str(path))
    if suffix == ".docx":
        return read_docx_to_text(str(path))
    raise ValueError(f"Format non supporte: {path}")

def export_input_texts() -> pd.DataFrame:
    inputs = [Path(p) for p in expand_inputs(INPUT_FILES, INPUT_PATTERNS)]
    rows: list[dict] = []
    seen_checksums: dict[str, str] = {}
    for src in inputs:
        checksum = file_sha256(src)
        if checksum in seen_checksums:
            rows.append({
                "source_path": str(src),
                "source_type": src.suffix.lower().lstrip("."),
                "output_txt": None,
                "chars": 0,
                "lines": 0,
                "checksum": checksum,
                "duplicate_of": seen_checksums[checksum],
                "status": "skipped_duplicate",
            })
            continue
        text = read_document_to_text(src)
        try:
            rel = src.relative_to(BASE_IN)
        except Exception:
            rel = Path(src.name)
        out = (BASE_OUT / rel).with_suffix(".txt")
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(text, encoding="utf-8")
        seen_checksums[checksum] = str(src)
        rows.append({
            "source_path": str(src),
            "source_type": src.suffix.lower().lstrip("."),
            "output_txt": str(out),
            "chars": len(text),
            "lines": len(text.splitlines()),
            "checksum": checksum,
            "duplicate_of": None,
            "status": "exported",
        })
    return pd.DataFrame(rows)

df_exports = export_input_texts()
df_exports.head(20)

{'ocr_fallback_used': 'data/in/MSO/remuneration_revalorisation/4. Instruction RDR AC-IDF du 7 juillet 2021.pdf', 'chars': 19236}


{'ocr_fallback_used': 'data/in/MSO/remuneration_revalorisation/5. Instruction RDR SD hors IDF du 22 mai 2024.pdf', 'chars': 14210}


,source_path,source_type,output_txt,chars,lines,checksum,duplicate_of,status
0,data/in/MSO/remuneration_revalorisation/4. Ins...,pdf,data/out/MSO/remuneration_revalorisation/4. In...,19236,317,6c9429113e57d02f59a8b3546a84c90036391dc5c014f2...,None,exported
1,data/in/MSO/remuneration_revalorisation/5. Ins...,pdf,data/out/MSO/remuneration_revalorisation/5. In...,14210,277,6598d90ca517c1ce9f067879e3cc8b67ac04e728f75d3c...,None,exported
2,data/in/MSO/remuneration_revalorisation/Cycle ...,pdf,data/out/MSO/remuneration_revalorisation/Cycle...,1086,61,3f199a463f008b882d12a05d22461d290092e220f92749...,None,exported
3,data/in/MSO/remuneration_revalorisation/Fixati...,pdf,data/out/MSO/remuneration_revalorisation/Fixat...,6543,187,ce288f67cff7d01f9b8586cf3318fb59c25178245bd2c4...,None,exported


## Etape 4 - Heuristiques de parsing et routage documentaire

Cette partie transforme le texte brut en structure exploitable. Elle definit :

- la detection des titres
- le nettoyage des tables des matieres
- la detection des FAQ
- la detection des processus
- la detection des matrices/tabulaires
- les parseurs specialises par type de document

C'est ici que se trouve l'intelligence metier du notebook. Si un document MSO est mal compris, le diagnostic se fait en general ici : soit le mauvais `document_mode` est choisi, soit le bon parseur ne decoupe pas correctement les sections.

In [5]:
TOC_START_PAT = re.compile(r"^table des matieres$", re.IGNORECASE)
TOC_ENTRY_PAT = re.compile(
    r"^(?:[IVXLC]+-|[A-Z]\.|\d+\.)?.{3,}\.{5,}\s*\d+\s*$",
    re.IGNORECASE,
)
HEADING_PATTERNS = [
    (1, re.compile(r"^\((?P<label>Titre)\)\s*(?P<title>.+)$", re.IGNORECASE)),
    (2, re.compile(r"^\((?P<label>Intertitre)\)\s*(?P<title>.+)$", re.IGNORECASE)),
    (1, re.compile(r"^(?P<label>[IVXLC]+-)\s*(?P<title>.+)$")),
    (2, re.compile(r"^(?P<label>[A-Z]\.)\s*(?P<title>.+)$")),
    (3, re.compile(r"^(?P<label>\d+\.[a-z])\s*(?P<title>.+)$")),
    (3, re.compile(r"^(?P<label>\d+\.)\s*(?P<title>.+)$")),
    (4, re.compile(r"^(?P<label>[a-z]\.)\s*(?P<title>.+)$")),
    (4, re.compile(r"^(?P<label>[✓✔☑])\s*(?P<title>.+)$")),
]

def strip_table_of_contents(text: str) -> str:
    lines = text.split("\n")
    out: list[str] = []
    in_toc = False
    toc_hits = 0
    for ln in lines:
        s = ln.strip()
        if TOC_START_PAT.match(s):
            in_toc = True
            toc_hits = 0
            continue
        if in_toc:
            if TOC_ENTRY_PAT.match(s) or s in {"", "[PAGE 2]", "[PAGE 3]"}:
                toc_hits += 1 if TOC_ENTRY_PAT.match(s) else 0
                continue
            if toc_hits >= 3:
                in_toc = False
            else:
                out.append(ln)
                continue
        out.append(ln)
    return normalize_text("\n".join(out))

def is_strong_synthetic_heading(clean: str) -> bool:
    letters = [c for c in clean if c.isalpha()]
    if not letters:
        return False
    upper_ratio = sum(1 for c in letters if c.isupper()) / len(letters)
    return upper_ratio >= 0.75

def detect_heading(line: str, synthetic_heading_mode: str = "normal"):
    clean = re.sub(r"\s+", " ", line).strip()
    clean = re.sub(r"\.{4,}\s*\d+$", "", clean).strip()
    if not clean or clean.startswith("[PAGE ") or clean.startswith("[SLIDE "):
        return None
    for level, pattern in HEADING_PATTERNS:
        m = pattern.match(clean)
        if m:
            title = m.group("title").strip(" -:\t")
            if title:
                return level, m.group("label"), title
    if clean.endswith("?") and len(clean) <= 140 and len(clean.split()) <= 14:
        return 2, "Q.", clean.strip(" ?")
    if (
        synthetic_heading_mode != "none"
        and (synthetic_heading_mode != "strict" or is_strong_synthetic_heading(clean))
        and len(clean) <= 90
        and 2 <= len(clean.split()) <= 10
        and clean[0].isupper()
        and not clean.startswith("-")
        and clean[-1] not in ".;,"
    ):
        return 2, "H.", clean.strip()
    return None

def looks_like_heading_continuation(line: str) -> bool:
    clean = re.sub(r"\s+", " ", line).strip()
    if not clean or detect_heading(clean, synthetic_heading_mode="none"):
        return False
    if len(clean) > 160:
        return False
    return clean[0].islower() or clean.startswith("(")

FAQ_NUMBERED_LINE_RE = re.compile(r"^(?P<label>\d{1,2})\s*\.\s*(?P<title>.+)$")
FAQ_TOC_LEADER_RE = re.compile(r"\.{5,}\s*\d+\s*$")

def has_toc_leader(line: str) -> bool:
    return bool(FAQ_TOC_LEADER_RE.search(re.sub(r"\s+", " ", line or "").strip()))

def is_faq_toc_entry_line(line: str) -> bool:
    clean = re.sub(r"\s+", " ", line or "").strip()
    if has_toc_leader(clean):
        return True
    return bool(FAQ_NUMBERED_LINE_RE.match(clean) and re.search(r"\?\s+\d{1,3}$", clean))

def strip_faq_number_prefix(line: str) -> tuple[str | None, str]:
    clean = re.sub(r"\s+", " ", line or "").strip()
    clean = re.sub(r"\.{5,}\s*\d+\s*$", "", clean).strip()
    m = FAQ_NUMBERED_LINE_RE.match(clean)
    if not m:
        return None, clean
    return m.group("label"), m.group("title").strip(" -:\t")

def is_probable_faq_section_line(line: str) -> bool:
    _, title = strip_faq_number_prefix(line)
    if not title or not FAQ_NUMBERED_LINE_RE.match(re.sub(r"\s+", " ", line or "").strip()) or is_faq_toc_entry_line(line):
        return False
    letters = [c for c in title if c.isalpha()]
    if not letters:
        return False
    upper_ratio = sum(1 for c in letters if c.isupper()) / len(letters)
    return upper_ratio >= 0.65 and len(title.split()) >= 2

def is_faq_question_line(line: str) -> bool:
    _, title = strip_faq_number_prefix(line)
    if not title or is_faq_toc_entry_line(line) or is_probable_faq_section_line(line):
        return False
    return "?" in title and len(title.split()) >= 3

def strip_faq_leading_toc(text: str) -> str:
    lines = text.split("\n")
    window = [ln.strip() for ln in lines[:240] if ln.strip()]
    if sum(1 for ln in window if is_faq_toc_entry_line(ln)) < 6:
        return text
    for i, raw in enumerate(lines[:600]):
        line = raw.strip()
        if not is_faq_question_line(line):
            continue
        page_start = 0
        for j in range(i - 1, -1, -1):
            if lines[j].strip().startswith("[PAGE "):
                page_start = j + 1
                break
        start = i
        for j in range(page_start, i):
            candidate = lines[j].strip()
            if is_probable_faq_section_line(candidate):
                start = j
                break
        return normalize_text("\n".join(lines[start:]))
    return text

def iter_faq_logical_lines(text: str) -> list[str]:
    cleaned = strip_faq_leading_toc(strip_table_of_contents(text))
    raw_lines = [ln.strip() for ln in cleaned.split("\n")]
    logical: list[str] = []
    i = 0
    while i < len(raw_lines):
        line = raw_lines[i]
        if not line or line.startswith("[PAGE ") or line.startswith("[SLIDE ") or (line.isdigit() and len(line) <= 3) or is_faq_toc_entry_line(line):
            i += 1
            continue
        if FAQ_NUMBERED_LINE_RE.match(line):
            parts = [line]
            j = i + 1
            if is_probable_faq_section_line(line):
                while j < len(raw_lines):
                    nxt = raw_lines[j].strip()
                    if not nxt or nxt.startswith("[PAGE ") or FAQ_NUMBERED_LINE_RE.match(nxt) or is_faq_toc_entry_line(nxt):
                        break
                    parts.append(nxt)
                    j += 1
            else:
                while "?" not in " ".join(parts) and j < len(raw_lines):
                    nxt = raw_lines[j].strip()
                    if not nxt or nxt.startswith("[PAGE ") or FAQ_NUMBERED_LINE_RE.match(nxt) or is_faq_toc_entry_line(nxt):
                        break
                    parts.append(nxt)
                    j += 1
            logical.append(normalize_text(" ".join(parts)))
            i = j
            continue
        logical.append(line)
        i += 1
    return logical

def looks_like_faq_text(text: str, source_name: str = "") -> bool:
    source_hint = "faq" in normalize_token_text(source_name)
    text_hint = "faq" in normalize_token_text(text[:1200])
    lines = iter_faq_logical_lines(text)
    question_hits = sum(1 for ln in lines if is_faq_question_line(ln))
    section_hits = sum(1 for ln in lines if is_probable_faq_section_line(ln))
    return question_hits >= 5 and (source_hint or text_hint or (question_hits >= 10 and section_hits >= 2))

def infer_user_question(section_title: str, section_path: str) -> str:
    title = re.sub(r"\s+", " ", section_title).strip(" .")
    if not title:
        return "Quelles sont les regles applicables dans cette section ?"
    if title.lower().startswith(("le ", "la ", "les ", "l'")):
        return f"Quelles sont les regles relatives a {title.lower()} ?"
    if any(token in title.lower() for token in ["procedure", "renouvellement", "recrutement", "conge", "temps partiel", "remuneration"]):
        return f"Quelle est la procedure ou les regles concernant {title.lower()} ?"
    if section_path:
        return f"Que faut-il savoir sur {title.lower()} dans le cadre de {section_path.lower()} ?"
    return f"Que faut-il savoir sur {title.lower()} ?"

def hard_wrap(text: str, max_chars: int, overlap: int) -> list[str]:
    res: list[str] = []
    i = 0
    step = max(1, max_chars - overlap)
    while i < len(text):
        res.append(text[i:i + max_chars])
        i += step
    return res

def split_on_paragraphs(text: str, max_chars: int = 1200, overlap: int = 200) -> list[str]:
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    out: list[str] = []
    buf = ""
    for p in paras:
        extra = 2 if buf else 0
        if len(buf) + extra + len(p) <= max_chars:
            buf = f"{buf}\n\n{p}" if buf else p
        else:
            if buf:
                out.append(buf)
            if len(p) > max_chars:
                out.extend(hard_wrap(p, max_chars, overlap))
                buf = ""
            else:
                buf = p
    if buf:
        out.append(buf)
    return out

@dataclass
class SectionBlock:
    qa_id: str
    parent_qa_id: str | None
    parent_section_path: str | None
    section_path: str
    section_index: int
    heading_level: int
    section_title: str
    pseudo_question: str
    answer: str
    source_name: str
    thematique: str

@dataclass
class Chunk:
    hash_id: str
    qa_id: str
    parent_qa_id: str | None
    role: str
    section_path: str
    chunk_index: int
    text: str
    chunk_text: str
    source_name: str
    lang: str = "fr"
    thematique: str = ""
    source: str = "MSO"
    short_id: str = "MSO"
    references_juridiques: list[dict] | list[str] | None = None
    section_id: str | None = None
    source_document_id: str | None = None
    embedding_bge_scw: list[float] | None = None

PROCESS_ACTOR_HINTS = (
    "sgcd", "dreets", "ddets", "drhm", "cbcm", "dgfip", "renoirh", "prefet", "agent", "services deconcentres", "service recruteur",
)
PROCESS_STEP_HINTS = (
    "publication", "selection", "production", "signature", "depot", "validation", "creation", "mise en paie", "pre liquidation", "simulation", "recrutement",
)
PROCESS_BRANCH_HINTS = {
    "si respect rdr": "Respect du RDR",
    "respect rdr": "Respect du RDR",
    "si hors rdr": "Hors RDR",
    "hors rdr": "Hors RDR",
    "si refus cbcm": "Refus du CBCM",
    "si visa cbcm": "Visa du CBCM",
}
BOILERPLATE_LINES = {
    "secrétariat général", "secrétariat general", "direction des ressources humaines", "c1 - public",
}

def normalize_token_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", text or "")
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    return re.sub(r"\s+", " ", text).strip()

def is_boilerplate_line(line: str) -> bool:
    norm = normalize_token_text(line)
    if not norm:
        return True
    if norm in BOILERPLATE_LINES:
        return True
    if re.fullmatch(r"\d{1,2}/\d{2}/\d{4}", norm):
        return True
    if norm.isdigit() and len(norm) <= 3:
        return True
    return False

def merge_process_fragments(lines: list[str]) -> list[str]:
    merged: list[str] = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue
        if line.startswith("[PAGE ") or line.startswith("[SLIDE "):
            merged.append(line)
            i += 1
            continue
        parts = [line]
        if len(line.split()) <= 3:
            j = i + 1
            while j < len(lines):
                nxt = lines[j].strip()
                if not nxt or nxt.startswith("[PAGE ") or nxt.startswith("[SLIDE "):
                    break
                if len(nxt.split()) > 3 or len(parts) >= 3:
                    break
                parts.append(nxt)
                j += 1
            merged.append(" ".join(parts))
            i = j
            continue
        merged.append(line)
        i += 1
    return merged

def canonical_branch_label(line: str) -> str | None:
    norm = normalize_token_text(line)
    return PROCESS_BRANCH_HINTS.get(norm)

def canonical_actor_label(line: str) -> str | None:
    norm = normalize_token_text(line)
    if any(hint in norm for hint in PROCESS_ACTOR_HINTS):
        return line.strip()
    return None

def is_process_step_title(line: str) -> bool:
    norm = normalize_token_text(line)
    if is_boilerplate_line(line) or canonical_branch_label(line) or canonical_actor_label(line):
        return False
    if len(line) > 110 or len(line.split()) < 2 or len(line.split()) > 12:
        return False
    if line.startswith("-") or line.endswith(":"):
        return False
    if any(hint in norm for hint in PROCESS_STEP_HINTS):
        return True
    return line[0].isupper() and line[-1] not in ".;,"

TABLE_SECTION_TITLES = {
    "propositions financières": "Propositions financières",
    "signature des contrats et avenants": "Signature des contrats et avenants",
    "fin du contrat": "Fin du contrat",
    "modalités de service": "Modalités de service",
    "formation et concours": "Formation et concours",
    "absences et congés": "Absences et congés",
    "disciplinaire": "Disciplinaire",
    "notice": "Notice",
}
TABLE_ENTITY_LABELS = (
    "Déconcentré",
    "DRH-BPECO",
    "DRH-BPECO / CBCM",
    "CBCM",
)

def canonical_table_entity(line: str) -> str | None:
    stripped = re.sub(r"\s+", " ", line).strip(" -")
    norm = normalize_token_text(stripped)
    for label in TABLE_ENTITY_LABELS:
        if normalize_token_text(label) == norm:
            return label
    return None

TABLE_SECTION_TITLES_NORM_MAP = {normalize_token_text(k): v for k, v in TABLE_SECTION_TITLES.items()}
TABLE_SECTION_TITLES_NORM = set(TABLE_SECTION_TITLES_NORM_MAP.keys())

def is_table_section_heading(line: str) -> bool:
    norm = normalize_token_text(line)
    return norm in TABLE_SECTION_TITLES_NORM

def canonical_table_section_heading(line: str) -> str | None:
    norm = normalize_token_text(line)
    if norm in TABLE_SECTION_TITLES_NORM_MAP:
        return TABLE_SECTION_TITLES_NORM_MAP[norm]
    for key, label in TABLE_SECTION_TITLES_NORM_MAP.items():
        if norm.startswith(key + " ") or norm.startswith(key + " •"):
            return label
    return None

def split_table_row(line: str) -> tuple[str, str, str] | None:
    compact = re.sub(r"\s+", " ", line).strip()
    m = re.match(r"^(?P<act>.+?)\s+(?P<alinea>(?:\d+°|-))\s+(?P<entity>Déconcentré|DRH-BPECO(?: / CBCM)?|CBCM)$", compact)
    if not m:
        return None
    return m.group("act").strip(), m.group("alinea").strip(), m.group("entity").strip()

def split_table_row_with_trailing_notice(line: str) -> tuple[tuple[str, str, str] | None, str | None]:
    compact = re.sub(r"\s+", " ", line).strip()
    m = re.match(r"^(?P<act>.+?)\s+(?P<alinea>(?:\d+°|-))\s+(?P<entity>Déconcentré|DRH-BPECO(?: / CBCM)?|CBCM)(?:\s+(?P<tail>.+))?$", compact)
    if not m:
        return None, None
    row = (m.group("act").strip(), m.group("alinea").strip(), m.group("entity").strip())
    tail = (m.group("tail") or "").strip() or None
    return row, tail

def infer_table_question(section_title: str, act_name: str, entity: str, alinea: str | None = None) -> str:
    act_norm = act_name.strip().rstrip(" .")
    entity_norm = entity.strip()
    if alinea and alinea != "-":
        return f"Quelle entite de gestion est competente pour {act_norm.lower()} au titre de l'alinea {alinea} ?"
    return f"Quelle entite de gestion est competente pour {act_norm.lower()} dans la rubrique {section_title.lower()} ?"

def detect_document_mode(text: str, source_name: str = "") -> str:
    cleaned = strip_table_of_contents(text)
    if looks_like_table_matrix_text(cleaned):
        return "table_matrix"
    if looks_like_faq_text(cleaned, source_name):
        return "faq"
    lines = [ln.strip() for ln in cleaned.split("\n") if ln.strip() and not ln.startswith("[PAGE ") and not ln.startswith("[SLIDE ")]
    if not lines:
        return "guide"
    guide_heading_hits = sum(1 for ln in lines if (heading := detect_heading(ln)) and heading[1] not in {"Q.", "H."})
    merged = merge_process_fragments(lines)
    short_ratio = sum(1 for ln in merged if len(ln.split()) <= 4) / max(1, len(merged))
    long_lines = sum(1 for ln in merged if len(ln.split()) >= 12)
    process_hits = sum(1 for ln in merged if is_process_step_title(ln) or canonical_branch_label(ln) or canonical_actor_label(ln))
    explicit_process = any(token in normalize_token_text(source_name) for token in ("processus", "logigramme")) or "logigramme" in normalize_token_text(cleaned[:1500])
    if guide_heading_hits >= 2 and not explicit_process:
        return "guide"
    if explicit_process or (short_ratio >= 0.35 and process_hits >= 8 and long_lines <= max(6, len(merged) // 6)):
        return "process"
    return "guide"

def infer_process_question(step_title: str, section_path: str, branch_label: str | None = None, actor_label: str | None = None) -> str:
    title = re.sub(r"\s+", " ", step_title).strip(" .")
    if branch_label and actor_label:
        return f"Que doit faire {actor_label.lower()} pour {title.lower()} dans le cas {branch_label.lower()} ?"
    if branch_label:
        return f"Que faut-il faire pour {title.lower()} dans le cas {branch_label.lower()} ?"
    if actor_label:
        return f"Quel est le role de {actor_label.lower()} pour {title.lower()} ?"
    if section_path:
        return f"Quelle est l'etape ou la regle concernant {title.lower()} dans le processus ?"
    return f"Que faut-il savoir sur {title.lower()} ?"

def parse_guide_blocks(text: str, source_name: str, thematique: str) -> list[SectionBlock]:
    cleaned = strip_table_of_contents(text)
    lines = cleaned.split("\n")
    content_lines = [ln.strip() for ln in lines if ln.strip() and not ln.strip().startswith(("[PAGE ", "[SLIDE "))]
    true_heading_hits = sum(1 for ln in content_lines if (heading := detect_heading(ln, synthetic_heading_mode="none")) and heading[1] not in {"Q.", "H."})
    structured_marker_hits = sum(1 for ln in content_lines if re.match(r"^\((?:titre|intertitre)\)", ln, re.IGNORECASE))
    synthetic_heading_mode = "none" if structured_marker_hits >= 2 else ("strict" if true_heading_hits >= 2 else "normal")
    blocks: list[SectionBlock] = []
    stack: list[dict] = []
    current: dict | None = None
    section_counter = 0

    def flush_current():
        nonlocal current
        if not current:
            return
        answer = normalize_text("\n".join(current["body"]))
        if answer:
            blocks.append(SectionBlock(current["qa_id"], current["parent_qa_id"], current["parent_section_path"], current["section_path"], current["section_index"], current["level"], current["title"], current["pseudo_question"], answer, source_name, thematique))
        current = None

    for raw in lines:
        line = raw.strip()
        if not line:
            if current:
                current["body"].append("")
            continue
        heading = detect_heading(line, synthetic_heading_mode=synthetic_heading_mode)
        if heading:
            flush_current()
            level, _, title = heading
            while stack and stack[-1]["level"] >= level:
                stack.pop()
            parent_qa_id = stack[-1]["qa_id"] if stack else None
            parent_section_path = " > ".join(x["title"] for x in stack) if stack else None
            section_counter += 1
            qa_id = sha1_u(f"{source_name}|guide|{section_counter}|{level}|{' > '.join(x['title'] for x in stack)}|{title}")
            section_titles = [x["title"] for x in stack] + [title]
            section_path = " > ".join(section_titles)
            pseudo_question = infer_user_question(title, section_path)
            current = {"qa_id": qa_id, "parent_qa_id": parent_qa_id, "parent_section_path": parent_section_path, "level": level, "title": title, "section_path": section_path, "section_index": section_counter, "pseudo_question": pseudo_question, "body": []}
            stack.append({"level": level, "title": title, "qa_id": qa_id})
            continue
        if current is None:
            continue
        if not current["body"] and looks_like_heading_continuation(line):
            current["title"] = normalize_text(f"{current['title']} {line.strip(' -')}")
            if stack and stack[-1]["qa_id"] == current["qa_id"]:
                stack[-1]["title"] = current["title"]
            parent_titles = [x["title"] for x in stack[:-1]]
            current["parent_section_path"] = " > ".join(parent_titles) if parent_titles else None
            current["section_path"] = " > ".join(parent_titles + [current["title"]])
            current["pseudo_question"] = infer_user_question(current["title"], current["section_path"])
            continue
        current["body"].append(line)

    flush_current()
    return blocks

def parse_faq_blocks(text: str, source_name: str, thematique: str) -> list[SectionBlock]:
    lines = iter_faq_logical_lines(text)
    blocks: list[SectionBlock] = []
    section_counter = 0
    current_section: str | None = None
    current: dict | None = None

    def flush_current():
        nonlocal current
        if not current:
            return
        answer = normalize_text("\n".join(current["body"]))
        if answer:
            blocks.append(SectionBlock(current["qa_id"], None, current["parent_section_path"], current["section_path"], current["section_index"], current["level"], current["title"], current["pseudo_question"], answer, source_name, thematique))
        current = None

    for line in lines:
        if is_probable_faq_section_line(line):
            flush_current()
            _, title = strip_faq_number_prefix(line)
            current_section = title
            continue
        if is_faq_question_line(line):
            flush_current()
            section_counter += 1
            _, title = strip_faq_number_prefix(line)
            question = title if title.endswith("?") else f"{title}?"
            section_path = f"{current_section} > {title}" if current_section else title
            qa_id = sha1_u(f"{source_name}|faq|{section_counter}|{current_section}|{title}")
            current = {"qa_id": qa_id, "parent_section_path": current_section, "level": 3, "title": title, "section_path": section_path, "section_index": section_counter, "pseudo_question": question, "body": []}
            continue
        if current:
            current["body"].append(line)

    flush_current()
    return blocks

def parse_process_blocks(text: str, source_name: str, thematique: str) -> list[SectionBlock]:
    cleaned = strip_table_of_contents(text)
    raw_lines = [ln for ln in cleaned.split("\n")]
    lines = [ln for ln in merge_process_fragments(raw_lines) if ln.strip()]
    blocks: list[SectionBlock] = []
    section_counter = 0
    current_heading: str | None = None
    current_branch: str | None = None
    current_step: dict | None = None

    def flush_step():
        nonlocal current_step
        if not current_step:
            return
        answer_lines = []
        if current_step.get("branch"):
            answer_lines.append(f"Branche: {current_step['branch']}")
        if current_step.get("actor"):
            answer_lines.append(f"Acteur principal: {current_step['actor']}")
        answer_lines.extend(current_step.get("body") or [])
        answer = normalize_text("\n".join(answer_lines)) or current_step["title"]
        blocks.append(SectionBlock(current_step["qa_id"], None, current_step.get("parent_section_path"), current_step["section_path"], current_step["section_index"], 2, current_step["title"], current_step["pseudo_question"], answer, source_name, thematique))
        current_step = None

    for line in lines:
        if is_boilerplate_line(line) or line.startswith("[PAGE ") or line.startswith("[SLIDE "):
            continue
        heading = detect_heading(line)
        if heading and heading[2].lower() not in {"q", "h"}:
            flush_step()
            current_heading = heading[2]
            current_branch = None
            continue
        branch_label = canonical_branch_label(line)
        if branch_label:
            flush_step()
            current_branch = branch_label
            continue
        actor_label = canonical_actor_label(line)
        if actor_label and current_step:
            current_step["actor"] = current_step.get("actor") or actor_label
            current_step["pseudo_question"] = infer_process_question(current_step["title"], current_step["section_path"], current_step.get("branch"), current_step.get("actor"))
            current_step["body"].append(actor_label)
            continue
        if is_process_step_title(line):
            flush_step()
            section_counter += 1
            parent_section_path = current_heading or None
            section_parts = [current_heading] if current_heading else []
            if current_branch:
                section_parts.append(current_branch)
            section_parts.append(line)
            section_path = " > ".join(part for part in section_parts if part)
            qa_id = sha1_u(f"{source_name}|process|{current_heading}|{current_branch}|{line}|{section_counter}")
            current_step = {
                "qa_id": qa_id,
                "title": line,
                "branch": current_branch,
                "actor": None,
                "body": [],
                "section_index": section_counter,
                "section_path": section_path,
                "parent_section_path": parent_section_path,
                "pseudo_question": infer_process_question(line, section_path, current_branch, None),
            }
            continue
        if current_step:
            current_step["body"].append(line)

    flush_step()
    return blocks

def parse_table_matrix_blocks(text: str, source_name: str, thematique: str) -> list[SectionBlock]:
    cleaned = strip_table_of_contents(text)
    lines = [ln.strip() for ln in cleaned.split("\n") if ln.strip() and not ln.startswith("[PAGE ") and not ln.startswith("[SLIDE ")]
    blocks: list[SectionBlock] = []
    section_counter = 0
    current_heading: str | None = None
    pending_parts: list[str] = []
    pending_alinea: str | None = None
    in_notice = False
    notice_lines: list[str] = []

    def flush_pending(entity: str | None = None):
        nonlocal pending_parts, pending_alinea, section_counter
        if not pending_parts:
            pending_alinea = None
            return
        act_name = normalize_text(" ".join(pending_parts))
        pending_parts = []
        if not act_name or not entity:
            pending_alinea = None
            return
        section_counter += 1
        effective_heading = current_heading or Path(source_name).stem
        section_path = f"{effective_heading} > {act_name}" if effective_heading else act_name
        qa_id = sha1_u(f"{source_name}|table|{effective_heading}|{act_name}|{pending_alinea}|{entity}|{section_counter}")
        answer_lines = [
            f"Rubrique: {effective_heading}",
            f"Type d'acte: {act_name}",
            f"Alinéa de référence: {pending_alinea or '-'}",
            f"Entité de gestion: {entity}",
        ]
        if notice_lines:
            answer_lines.append("Notice: " + " ".join(notice_lines[:2]))
        blocks.append(SectionBlock(qa_id, None, effective_heading if current_heading else None, section_path, section_counter, 2, act_name, infer_table_question(effective_heading, act_name, entity, pending_alinea), normalize_text("\n".join(answer_lines)), source_name, thematique))
        pending_alinea = None

    for line in lines:
        norm = normalize_token_text(line)
        if norm in {"type d'actes", "entite de gestion"}:
            continue
        if line == "Notice":
            flush_pending()
            in_notice = True
            continue
        if norm in {"type d'actes entite de gestion", "entite de gestion type d'actes"}:
            continue
        if in_notice and (line.startswith("•") or norm.startswith("la colonne") or norm.startswith("a gauche") or norm.startswith("les autres feuilles")):
            notice_lines.append(line.lstrip("• "))
            continue
        if norm.startswith("references :") or norm.startswith("arretes du") or norm.startswith("les actes devront etre") or norm == "attention !":
            continue
        direct_row, trailing_notice = split_table_row_with_trailing_notice(line)
        if direct_row:
            flush_pending()
            pending_parts = [direct_row[0]]
            pending_alinea = direct_row[1]
            flush_pending(direct_row[2])
            if trailing_notice and (trailing_notice.startswith("•") or trailing_notice.startswith("La colonne") or trailing_notice.startswith("A gauche") or trailing_notice.startswith("Les autres feuilles") or trailing_notice.startswith("niveau du")):
                notice_lines.append(trailing_notice.lstrip("• "))
            continue
        heading_label = canonical_table_section_heading(line)
        if heading_label and heading_label != current_heading:
            flush_pending()
            current_heading = heading_label
            in_notice = False
            remainder = line[len(heading_label):].strip(" :-•")
            if remainder and (remainder.startswith("La colonne") or remainder.startswith("A gauche") or remainder.startswith("Les autres feuilles") or remainder.startswith("•")):
                notice_lines.append(remainder.lstrip("• "))
            continue
        entity = canonical_table_entity(line)
        if entity:
            flush_pending(entity)
            continue
        if re.fullmatch(r"(?:\d+°|-)", line):
            pending_alinea = line
            continue
        if is_boilerplate_line(line):
            continue
        pending_parts.append(line)

    flush_pending()
    if not blocks:
        return parse_process_blocks(text, source_name, thematique)
    return blocks

def make_hash_id(source_name: str, qa_id: str, role: str, chunk_index: int, text: str) -> str:
    return sha1_u(f"{source_name}|{qa_id}|{role}|{chunk_index}|{text[:256]}")

def dedupe_section_blocks(blocks: list[SectionBlock]) -> list[SectionBlock]:
    deduped: list[SectionBlock] = []
    seen: set[tuple[str, str, str, str]] = set()
    for block in blocks:
        key = (block.source_name, block.section_path, block.pseudo_question, block.answer)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(block)
    for idx, block in enumerate(deduped, start=1):
        block.section_index = idx
    return deduped

def dedupe_chunks(rows: list[Chunk]) -> list[Chunk]:
    deduped: list[Chunk] = []
    seen: set[str] = set()
    for row in rows:
        if row.hash_id in seen:
            continue
        seen.add(row.hash_id)
        deduped.append(row)
    return deduped

def section_blocks_to_chunks(blocks: list[SectionBlock], *, doc_id: str, section_id_by_qa_id: dict[str, str]) -> list[Chunk]:
    rows: list[Chunk] = []
    for block in blocks:
        q = block.pseudo_question
        a = block.answer
        section_id = section_id_by_qa_id.get(block.qa_id)
        prefix = f"Titre: {block.section_title}\nSection: {block.section_path}"
        q_text = f"{prefix}\nQuestion utilisateur probable: {q}"
        q_hash = make_hash_id(block.source_name, block.qa_id, "Q_ONLY", 0, q_text)
        rows.append(Chunk(q_hash, block.qa_id, block.parent_qa_id, "Q_ONLY", block.section_path, 0, q_text, q_text, block.source_name, thematique=block.thematique, section_id=section_id, source_document_id=doc_id, references_juridiques=[]))

        composite = f"{prefix}\nQuestion utilisateur probable: {q}\n\nContenu:\n{a}"[:3000]
        comp_hash = make_hash_id(block.source_name, block.qa_id, "QA_COMPOSITE", 1, composite)
        rows.append(Chunk(comp_hash, block.qa_id, block.parent_qa_id or block.qa_id, "QA_COMPOSITE", block.section_path, 1, composite, composite, block.source_name, thematique=block.thematique, section_id=section_id, source_document_id=doc_id, references_juridiques=[]))

        next_idx = 2
        for piece in split_on_paragraphs(a, max_chars=1200, overlap=200):
            atomic = f"Titre: {block.section_title}\nSection: {block.section_path}\nQuestion utilisateur probable: {q}\n\nContenu:\n{piece}"
            atomic_hash = make_hash_id(block.source_name, block.qa_id, "A_ATOMIC", next_idx, atomic)
            rows.append(Chunk(atomic_hash, block.qa_id, block.parent_qa_id or block.qa_id, "A_ATOMIC", block.section_path, next_idx, atomic, atomic, block.source_name, thematique=block.thematique, section_id=section_id, source_document_id=doc_id, references_juridiques=[]))
            next_idx += 1
    return dedupe_chunks(rows)

## Etape 5 - Construction des datasets `documents / sections / chunks`

Cette cellule applique les parseurs aux `.txt` exportes et fabrique les trois niveaux d'artefacts du pipeline :

- `documents` : metadonnees documentaires
- `sections` : structure logique des sections
- `chunks` : unites RAG indexables

Chaque document est route vers un mode (`guide`, `faq`, `process`, `table_matrix`) puis converti en blocs, ensuite en chunks. Les IDs sont deterministes pour permettre le reprocessing sans destabiliser l'ingestion.

La sortie affichee permet de verifier rapidement :

- le mode choisi pour chaque document
- le nombre de sections et de chunks produits
- le format du texte final indexe

In [6]:
def build_chunks_dataset() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    txt_files = expand_inputs([], TXT_GLOB)
    exported_inputs = {
        str(Path(rec['output_txt'])): rec
        for rec in df_exports.to_dict(orient='records')
        if rec.get('output_txt')
    }
    all_chunks: list[Chunk] = []
    all_documents: list[dict] = []
    all_sections: list[dict] = []
    for path_str in txt_files:
        path = Path(path_str)
        source_meta = exported_inputs.get(str(path), {})
        source_path = Path(source_meta.get('source_path')) if source_meta.get('source_path') else None
        source_type = source_meta.get('source_type') or (source_path.suffix.lower().lstrip('.') if source_path else 'pdf')
        text = normalize_text(path.read_text(encoding="utf-8", errors="ignore"))
        thematique = compute_thematique(str(path))
        rel_txt = path.relative_to(BASE_OUT) if path.is_relative_to(BASE_OUT) else Path(path.name)
        doc_title = path.stem
        short_id = slugify_short_id(str(rel_txt.with_suffix('')))
        doc_id = stable_uuid_from_parts(MSO_NAMESPACE, 'mso', short_id, str(rel_txt))
        checksum = sha256_text(text)
        doc_mode = detect_document_mode(text, path.name)
        parser = {'process': parse_process_blocks, 'table_matrix': parse_table_matrix_blocks, 'faq': parse_faq_blocks}.get(doc_mode, parse_guide_blocks)
        blocks = parser(text, path.name, thematique)
        if not blocks and doc_mode in {'process', 'table_matrix'}:
            blocks = parse_guide_blocks(text, path.name, thematique)
        if not blocks and text:
            fallback_title = path.stem
            blocks = [
                SectionBlock(
                    qa_id=sha1_u(f"{path.name}|fallback"),
                    parent_qa_id=None,
                    parent_section_path=None,
                    section_path=fallback_title,
                    section_index=1,
                    heading_level=1,
                    section_title=fallback_title,
                    pseudo_question=infer_user_question(fallback_title, fallback_title),
                    answer=text,
                    source_name=path.name,
                    thematique=thematique,
                )
            ]
            doc_mode = 'fallback'
        blocks = dedupe_section_blocks(blocks)
        section_id_by_qa_id: dict[str, str] = {}
        for block in blocks:
            section_id_by_qa_id[block.qa_id] = stable_uuid_from_parts(MSO_NAMESPACE, doc_id, 'section', block.qa_id)
        doc_record = {
            'doc_id': doc_id,
            'source': 'mso',
            'source_url': None,
            'storage_path': None,
            'title': doc_title,
            'full_title': doc_title,
            'short_id': short_id,
            'publisher': 'MSO',
            'doc_type': 'Tableau RH' if doc_mode == 'table_matrix' else ('Processus RH' if doc_mode == 'process' else ('FAQ RH' if doc_mode == 'faq' else 'Guide RH')),
            'last_updated_date': None,
            'publication_date': None,
            'page_count': None,
            'lang': 'fr',
            'checksum': checksum,
            'parse_version': 'extract_pdf_MSO_v3',
            'parse_model': MODEL_NAME,
            'quality_flags': {'source_format': source_type, 'section_aware': True, 'parse_mode': doc_mode},
            'doc_markdown': text,
            'doc_markdown_raw': text,
            'doc_text_hash': checksum,
            'token_count': estimate_tokens(text),
            'char_count': len(text),
            'line_count': len(text.splitlines()),
            'metadata': {
                'local_source_path': str(source_path) if source_path else None,
                'local_txt_path': str(path),
                'thematique': thematique,
                'source_type': source_type,
            },
            'doc_structure': {
                'section_count': len(blocks),
                'max_section_level': max((b.heading_level for b in blocks), default=0),
                'types': [f'{doc_mode}_mso', source_type],
            },
            'legacy_doc_id': None,
            'created_at': utc_now_iso(),
            'updated_at': utc_now_iso(),
        }
        all_documents.append(doc_record)
        for block in blocks:
            section_markdown = f"## {block.section_title}\n\n{block.answer}".strip()
            all_sections.append({
                'section_id': section_id_by_qa_id[block.qa_id],
                'doc_id': doc_id,
                'section_index': block.section_index,
                'parent_section_id': None,
                'level': block.heading_level,
                'section_type': 'heading',
                'heading': block.section_title,
                'heading_path': block.section_path,
                'page_start': None,
                'page_end': None,
                'char_start': None,
                'char_end': None,
                'section_markdown': section_markdown,
                'token_count': estimate_tokens(section_markdown),
                'char_count': len(section_markdown),
                'teash': None,
                'doc_text_hash': checksum,
                'metadata': {'publisher': 'MSO', 'doc_short_id': short_id},
                'created_at': doc_record['created_at'],
                'updated_at': doc_record['updated_at'],
                'text_hash': sha256_text(section_markdown),
                'is_indexable': True,
                'references_juridiques': [],
            })
        chunks = section_blocks_to_chunks(blocks, doc_id=doc_id, section_id_by_qa_id=section_id_by_qa_id)
        for chunk in chunks:
            chunk.short_id = short_id
        all_chunks.extend(chunks)
        print(f"{path.name}: mode={doc_mode} type={source_type} {len(blocks)} sections -> {len(chunks)} chunks")

    out_path = Path(OUT_JSONL)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for chunk in all_chunks:
            f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")
    docs_path = Path(OUT_DOCS_JSONL)
    docs_path.parent.mkdir(parents=True, exist_ok=True)
    with docs_path.open('w', encoding='utf-8') as f:
        for rec in all_documents:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    sections_path = Path(OUT_SECTIONS_JSONL)
    sections_path.parent.mkdir(parents=True, exist_ok=True)
    with sections_path.open('w', encoding='utf-8') as f:
        for rec in all_sections:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f"JSONL saved to {out_path} ({len(all_chunks)} rows)")
    return pd.DataFrame([asdict(c) for c in all_chunks]), pd.DataFrame(all_documents), pd.DataFrame(all_sections)

df_chunks, df_documents, df_sections = build_chunks_dataset()
with pd.option_context("display.max_colwidth", 140):
    display(df_chunks[["source_name", "role", "section_path", "chunk_index", "text"]].head(20))

4. Instruction RDR AC-IDF du 7 juillet 2021.txt: mode=guide type=pdf 10 sections -> 45 chunks
5. Instruction RDR SD hors IDF du 22 mai 2024.txt: mode=guide type=pdf 33 sections -> 104 chunks
Cycle de la paie_v2.txt: mode=process type=pdf 1 sections -> 3 chunks
Fixation à l'embauche et évolution de la rémunération d'un agent contractuel.txt: mode=guide type=pdf 25 sections -> 75 chunks
JSONL saved to data/out/chunked/mso_remuneration_revalorisation_chunks_qna.jsonl (227 rows)


,source_name,role,section_path,chunk_index,text
0,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,Q_ONLY,Instruction relative à la rémunération des agents contractuels,0,Titre: Instruction relative à la rémunération des agents contractuels\nSection: Instruction relative à la rémunération des agents contra...
1,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,QA_COMPOSITE,Instruction relative à la rémunération des agents contractuels,1,Titre: Instruction relative à la rémunération des agents contractuels\nSection: Instruction relative à la rémunération des agents contra...
2,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,Instruction relative à la rémunération des agents contractuels,2,Titre: Instruction relative à la rémunération des agents contractuels\nSection: Instruction relative à la rémunération des agents contra...
3,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,Q_ONLY,"En outre, ce dispositif vise 4 objectifs :",0,"Titre: En outre, ce dispositif vise 4 objectifs :\nSection: En outre, ce dispositif vise 4 objectifs :\nQuestion utilisateur probable: Q..."
4,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,QA_COMPOSITE,"En outre, ce dispositif vise 4 objectifs :",1,"Titre: En outre, ce dispositif vise 4 objectifs :\nSection: En outre, ce dispositif vise 4 objectifs :\nQuestion utilisateur probable: Q..."
5,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,"En outre, ce dispositif vise 4 objectifs :",2,"Titre: En outre, ce dispositif vise 4 objectifs :\nSection: En outre, ce dispositif vise 4 objectifs :\nQuestion utilisateur probable: Q..."
6,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,"En outre, ce dispositif vise 4 objectifs :",3,"Titre: En outre, ce dispositif vise 4 objectifs :\nSection: En outre, ce dispositif vise 4 objectifs :\nQuestion utilisateur probable: Q..."
7,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,Q_ONLY,METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL,0,Titre: METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL\nSection: METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL\nQuestion utili...
8,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,QA_COMPOSITE,METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL,1,Titre: METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL\nSection: METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL\nQuestion utili...
9,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL,2,Titre: METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL\nSection: METHODOLOGIE APPLIQUEE A LA CREATION DU REFERENTIEL\nQuestion utili...


## Etape 6 - Generation des embeddings

Cette cellule prend les chunks produits et calcule les vecteurs utilises au retrieval.

Deux familles d'embeddings peuvent etre produites :

- `embedding_m3` via le modele local `BAAI/bge-m3`
- `embedding_bge_scw` via l'API Scaleway si elle est activee

Artefacts produits :

- JSONL avec embeddings
- fichier `.npy`
- fichier `.parquet`

Point de vigilance : si cette cellule termine mais que certaines colonnes d'embeddings sont vides, le probleme ne vient plus du parsing mais de la generation vectorielle ou de la configuration API.

In [7]:
def format_passage(text: str) -> str:
    if MODEL_NAME.startswith("intfloat/multilingual-e5"):
        return f"passage: {text or ''}"
    return text or ""

def add_embeddings() -> pd.DataFrame:
    rows = [json.loads(line) for line in Path(OUT_JSONL).read_text(encoding="utf-8").splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    assert not df.empty, "Aucun chunk a embedder."
    corpus = [format_passage(t) for t in df["chunk_text"].astype(str).tolist()]
    vecs_all = []
    for i in range(0, len(corpus), BATCH_SIZE):
        batch = corpus[i:i + BATCH_SIZE]
        vecs = model.encode(
            batch,
            batch_size=len(batch),
            convert_to_numpy=True,
            normalize_embeddings=NORMALIZE,
            show_progress_bar=True,
        )
        vecs_all.append(vecs)
    emb = np.vstack(vecs_all).astype(np.float32)
    df[EMBED_COL] = [v.tolist() for v in emb]
    if GENERATE_BGE_SCW:
        bge_vectors: list[list[float]] = []
        for i in range(0, len(corpus), SCALEWAY_BGE_BATCH_SIZE):
            batch = corpus[i:i + SCALEWAY_BGE_BATCH_SIZE]
            print({"embedding_bge_scw_batch_start": i, "batch_size": len(batch)})
            for text in batch:
                bge_vectors.append(scaleway_embed_text(text))
        df["embedding_bge_scw"] = bge_vectors
    elif "embedding_bge_scw" not in df.columns:
        df["embedding_bge_scw"] = None
    if "references_juridiques" not in df.columns:
        df["references_juridiques"] = None
    if "section_id" not in df.columns:
        df["section_id"] = None
    if "source_document_id" not in df.columns:
        df["source_document_id"] = None
    Path(OUT_JSONL_WITH_EMB).parent.mkdir(parents=True, exist_ok=True)
    with open(OUT_JSONL_WITH_EMB, "w", encoding="utf-8") as f:
        for rec in df.to_dict(orient="records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    np.save(OUT_NPY, emb)
    parquet_df = df.copy()
    parquet_df[EMBED_COL] = parquet_df[EMBED_COL].apply(json.dumps)
    if "embedding_bge_scw" in parquet_df.columns:
        parquet_df["embedding_bge_scw"] = parquet_df["embedding_bge_scw"].apply(lambda v: json.dumps(v) if isinstance(v, list) else None)
    parquet_df.to_parquet(OUT_PARQUET, index=False)
    print({"jsonl_with_emb": OUT_JSONL_WITH_EMB, "npy": OUT_NPY, "parquet": OUT_PARQUET, "shape": emb.shape, "embedding_bge_scw_generated": bool(GENERATE_BGE_SCW)})
    return df

df_emb = add_embeddings()
df_emb[["source_name", "role", "section_path", "chunk_index"]].head(20)

Batches:   0%|                                            | 0/1 [00:00<?, ?it/s]

Batches: 100%|████████████████████████████████████| 1/1 [00:20<00:00, 20.17s/it]

Batches: 100%|████████████████████████████████████| 1/1 [00:20<00:00, 20.18s/it]

Batches:   0%|                                            | 0/1 [00:00<?, ?it/s]

Batches: 100%|████████████████████████████████████| 1/1 [00:16<00:00, 16.09s/it]

...

{'embedding_bge_scw_batch_start': 192, 'batch_size': 16}


{'embedding_bge_scw_batch_start': 208, 'batch_size': 16}


{'embedding_bge_scw_batch_start': 224, 'batch_size': 3}


{'jsonl_with_emb': './data/out/mso_remuneration_revalorisation_chunks_with_emb.jsonl', 'npy': './data/out/mso_remuneration_revalorisation_chunks.npy', 'parquet': './data/out/mso_remuneration_revalorisation_chunks.parquet', 'shape': (227, 1024), 'embedding_bge_scw_generated': True}


,source_name,role,section_path,chunk_index
0,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,Q_ONLY,Instruction relative à la rémunération des age...,0
1,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,QA_COMPOSITE,Instruction relative à la rémunération des age...,1
2,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,Instruction relative à la rémunération des age...,2
3,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,Q_ONLY,"En outre, ce dispositif vise 4 objectifs :",0
4,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,QA_COMPOSITE,"En outre, ce dispositif vise 4 objectifs :",1
5,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,"En outre, ce dispositif vise 4 objectifs :",2
6,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,"En outre, ce dispositif vise 4 objectifs :",3
7,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,Q_ONLY,METHODOLOGIE APPLIQUEE A LA CREATION DU REFERE...,0
8,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,QA_COMPOSITE,METHODOLOGIE APPLIQUEE A LA CREATION DU REFERE...,1
9,4. Instruction RDR AC-IDF du 7 juillet 2021.txt,A_ATOMIC,METHODOLOGIE APPLIQUEE A LA CREATION DU REFERE...,2


## Etape 7 - Schema SQL et upsert en base

Cette partie declare les tables cibles et la fonction `upsert_to_db()`.

Elle cree ou alimente :

- `public.rag_documents`
- `public.rag_sections`
- `public.documents`
- `public.rag_chunks_mso`

Le point important est le mode `MSO_REPLACE_EXISTING_DOCS=1` : il permet de recharger proprement un meme lot sans accumuler des doublons logiques.

Quand quelque chose parait correct localement mais n'apparait pas en base, le probleme est souvent dans cette etape : mauvais DSN, mauvaise table, transaction incomplete ou document remplace partiellement.

In [8]:
CREATE_DOCUMENTS_SQL = """
CREATE TABLE IF NOT EXISTS \"public\".\"rag_documents\" (
    doc_id UUID PRIMARY KEY,
    source TEXT,
    source_url TEXT,
    storage_path TEXT,
    title TEXT,
    full_title TEXT,
    short_id VARCHAR(64),
    publisher TEXT,
    doc_type TEXT,
    last_updated_date DATE,
    publication_date DATE,
    page_count INTEGER,
    lang TEXT,
    checksum TEXT,
    parse_version TEXT,
    parse_model TEXT,
    quality_flags JSONB,
    doc_markdown TEXT,
    doc_text_hash TEXT,
    metadata JSONB,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    doc_markdown_raw TEXT,
    token_count INTEGER,
    char_count INTEGER,
    line_count INTEGER,
    doc_structure JSONB,
    legacy_doc_id UUID
);
CREATE INDEX IF NOT EXISTS idx_rag_documents_short_id ON \"public\".\"rag_documents\" (short_id);
"""

CREATE_SECTIONS_SQL = """
CREATE TABLE IF NOT EXISTS \"public\".\"rag_sections\" (
    section_id UUID PRIMARY KEY,
    doc_id UUID,
    section_index INTEGER,
    parent_section_id UUID,
    level SMALLINT,
    section_type TEXT,
    heading TEXT,
    heading_path TEXT,
    page_start INTEGER,
    page_end INTEGER,
    char_start INTEGER,
    char_end INTEGER,
    section_markdown TEXT,
    token_count INTEGER,
    char_count INTEGER,
    teash TEXT,
    doc_text_hash TEXT,
    metadata JSONB,
    created_at TIMESTAMPTZ,
    updated_at TIMESTAMPTZ,
    text_hash TEXT,
    is_indexable BOOLEAN,
    references_juridiques JSONB
);
CREATE INDEX IF NOT EXISTS idx_rag_sections_doc_id ON \"public\".\"rag_sections\" (doc_id);
CREATE INDEX IF NOT EXISTS idx_rag_sections_refs ON \"public\".\"rag_sections\" USING GIN (references_juridiques);
"""

CREATE_LEGACY_DOCUMENTS_SQL = """
CREATE TABLE IF NOT EXISTS \"public\".\"documents\" (
    id UUID PRIMARY KEY,
    filename TEXT,
    mime_type TEXT,
    size_bytes BIGINT,
    sha256_hex TEXT,
    source_name TEXT,
    created_at TIMESTAMPTZ,
    data BYTEA
);
"""

CREATE_TABLE_SQL = f"""
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE IF NOT EXISTS \"{SCHEMA}\".\"{TABLE}\" (
    hash_id VARCHAR(64) PRIMARY KEY,
    qa_id TEXT,
    parent_qa_id TEXT,
    source_name VARCHAR(255),
    section_path TEXT,
    role TEXT,
    chunk_index INTEGER,
    text TEXT,
    chunk_text TEXT,
    lang TEXT DEFAULT 'fr',
    thematique TEXT,
    source TEXT,
    short_id VARCHAR(64),
    references_juridiques JSONB,
    section_id UUID,
    source_document_id UUID,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    text_tsv tsvector GENERATED ALWAYS AS (
        to_tsvector('french', coalesce(section_path, '') || ' ' || coalesce(chunk_text, ''))
    ) STORED,
    embedding_m3 vector(1024),
    embedding_bge_scw vector(3584)
);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_short_id ON \"{SCHEMA}\".\"{TABLE}\" (short_id);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_section_id ON \"{SCHEMA}\".\"{TABLE}\" (section_id);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_tsv ON \"{SCHEMA}\".\"{TABLE}\" USING GIN (text_tsv);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_{EMBED_COL} ON \"{SCHEMA}\".\"{TABLE}\" USING ivfflat ({EMBED_COL} vector_cosine_ops) WITH (lists = 100);
"""

UPSERT_LEGACY_DOCUMENTS_SQL = """
INSERT INTO \"public\".\"documents\"
(id, filename, mime_type, size_bytes, sha256_hex, source_name, created_at, data)
VALUES
(%(id)s::uuid, %(filename)s, %(mime_type)s, %(size_bytes)s, %(sha256_hex)s, %(source_name)s, %(created_at)s::timestamptz, %(data)s)
ON CONFLICT (id) DO UPDATE SET
    filename = EXCLUDED.filename,
    mime_type = EXCLUDED.mime_type,
    size_bytes = EXCLUDED.size_bytes,
    sha256_hex = EXCLUDED.sha256_hex,
    source_name = EXCLUDED.source_name,
    data = EXCLUDED.data;
"""

UPSERT_DOCUMENTS_SQL = """
INSERT INTO \"public\".\"rag_documents\"
(doc_id, source, source_url, storage_path, title, full_title, publisher, doc_type, last_updated_date, publication_date, page_count, lang, checksum, parse_version, parse_model, quality_flags, doc_markdown, doc_text_hash, metadata, created_at, updated_at, doc_markdown_raw, token_count, char_count, line_count, short_id, doc_structure, legacy_doc_id)
VALUES
(%(doc_id)s::uuid, %(source)s, %(source_url)s, %(storage_path)s, %(title)s, %(full_title)s, %(publisher)s, %(doc_type)s, %(last_updated_date)s, %(publication_date)s, %(page_count)s, %(lang)s, %(checksum)s, %(parse_version)s, %(parse_model)s, %(quality_flags)s::jsonb, %(doc_markdown)s, %(doc_text_hash)s, %(metadata)s::jsonb, %(created_at)s::timestamp, %(updated_at)s::timestamp, %(doc_markdown_raw)s, %(token_count)s, %(char_count)s, %(line_count)s, %(short_id)s, %(doc_structure)s::jsonb, %(legacy_doc_id)s::uuid)
ON CONFLICT (doc_id) DO UPDATE SET
    source = EXCLUDED.source,
    source_url = EXCLUDED.source_url,
    storage_path = EXCLUDED.storage_path,
    title = EXCLUDED.title,
    full_title = EXCLUDED.full_title,
    publisher = EXCLUDED.publisher,
    doc_type = EXCLUDED.doc_type,
    last_updated_date = EXCLUDED.last_updated_date,
    publication_date = EXCLUDED.publication_date,
    page_count = EXCLUDED.page_count,
    lang = EXCLUDED.lang,
    checksum = EXCLUDED.checksum,
    parse_version = EXCLUDED.parse_version,
    parse_model = EXCLUDED.parse_model,
    quality_flags = EXCLUDED.quality_flags,
    doc_markdown = EXCLUDED.doc_markdown,
    doc_text_hash = EXCLUDED.doc_text_hash,
    metadata = EXCLUDED.metadata,
    updated_at = EXCLUDED.updated_at,
    doc_markdown_raw = EXCLUDED.doc_markdown_raw,
    token_count = EXCLUDED.token_count,
    char_count = EXCLUDED.char_count,
    line_count = EXCLUDED.line_count,
    short_id = EXCLUDED.short_id,
    doc_structure = EXCLUDED.doc_structure,
    legacy_doc_id = EXCLUDED.legacy_doc_id;
"""

UPSERT_SECTIONS_SQL = """
INSERT INTO \"public\".\"rag_sections\"
(section_id, doc_id, section_index, parent_section_id, level, section_type, heading, heading_path, page_start, page_end, char_start, char_end, section_markdown, token_count, char_count, teash, doc_text_hash, metadata, created_at, updated_at, text_hash, is_indexable, references_juridiques)
VALUES
(%(section_id)s::uuid, %(doc_id)s::uuid, %(section_index)s, %(parent_section_id)s::uuid, %(level)s, %(section_type)s, %(heading)s, %(heading_path)s, %(page_start)s, %(page_end)s, %(char_start)s, %(char_end)s, %(section_markdown)s, %(token_count)s, %(char_count)s, %(teash)s, %(doc_text_hash)s, %(metadata)s::jsonb, %(created_at)s::timestamptz, %(updated_at)s::timestamptz, %(text_hash)s, %(is_indexable)s, %(references_juridiques)s::jsonb)
ON CONFLICT (section_id) DO UPDATE SET
    doc_id = EXCLUDED.doc_id,
    section_index = EXCLUDED.section_index,
    parent_section_id = EXCLUDED.parent_section_id,
    section_type = EXCLUDED.section_type,
    level = EXCLUDED.level,
    heading = EXCLUDED.heading,
    heading_path = EXCLUDED.heading_path,
    page_start = EXCLUDED.page_start,
    page_end = EXCLUDED.page_end,
    char_start = EXCLUDED.char_start,
    char_end = EXCLUDED.char_end,
    section_markdown = EXCLUDED.section_markdown,
    token_count = EXCLUDED.token_count,
    char_count = EXCLUDED.char_count,
    teash = EXCLUDED.teash,
    doc_text_hash = EXCLUDED.doc_text_hash,
    metadata = EXCLUDED.metadata,
    created_at = EXCLUDED.created_at,
    updated_at = EXCLUDED.updated_at,
    text_hash = EXCLUDED.text_hash,
    is_indexable = EXCLUDED.is_indexable,
    references_juridiques = EXCLUDED.references_juridiques;
"""

UPSERT_SQL = f"""
INSERT INTO \"{SCHEMA}\".\"{TABLE}\"
(hash_id, qa_id, parent_qa_id, source_name, section_path, role, chunk_index, text, chunk_text, lang, thematique, short_id, source, references_juridiques, section_id, source_document_id, embedding_m3, embedding_bge_scw)
VALUES
(%(hash_id)s, %(qa_id)s, %(parent_qa_id)s, %(source_name)s, %(section_path)s, %(role)s, %(chunk_index)s, %(text)s, %(chunk_text)s, %(lang)s, %(thematique)s, %(short_id)s, %(source)s, %(references_juridiques)s::jsonb, %(section_id)s::uuid, %(source_document_id)s::uuid, %(embedding_m3)s::vector, %(embedding_bge_scw)s::vector)
ON CONFLICT (hash_id) DO UPDATE SET
    qa_id = EXCLUDED.qa_id,
    parent_qa_id = EXCLUDED.parent_qa_id,
    source_name = EXCLUDED.source_name,
    section_path = EXCLUDED.section_path,
    role = EXCLUDED.role,
    chunk_index = EXCLUDED.chunk_index,
    text = EXCLUDED.text,
    chunk_text = EXCLUDED.chunk_text,
    lang = EXCLUDED.lang,
    thematique = EXCLUDED.thematique,
    short_id = EXCLUDED.short_id,
    source = EXCLUDED.source,
    references_juridiques = EXCLUDED.references_juridiques,
    section_id = EXCLUDED.section_id,
    source_document_id = EXCLUDED.source_document_id,
    embedding_m3 = EXCLUDED.embedding_m3,
    embedding_bge_scw = EXCLUDED.embedding_bge_scw,
    updated_at = CURRENT_TIMESTAMP;
"""

def vec_to_pgvector(v: Iterable[float]) -> str:
    vals = list(v)
    return "[" + ",".join(f"{float(x):.7f}" for x in vals) + "]"

def make_legacy_document_payload(doc_record: dict) -> dict:
    meta = doc_record.get('metadata') or {}
    source_path = Path(meta.get('local_source_path') or meta.get('local_pdf_path') or meta.get('original_local_source_path') or meta.get('original_local_pdf_path') or '')
    if not str(source_path) or str(source_path) == '.' or not source_path.exists() or source_path.is_dir():
        return None
    file_bytes = source_path.read_bytes()
    file_sha = hashlib.sha256(file_bytes).hexdigest()
    legacy_doc_id = stable_uuid_from_parts(MSO_NAMESPACE, 'legacy_source', source_path.name, file_sha)
    mime_type = {
        '.pdf': 'application/pdf',
        '.pptx': 'application/vnd.openxmlformats-officedocument.presentationml.presentation',
    }.get(source_path.suffix.lower(), 'application/octet-stream')
    return {
        'id': legacy_doc_id,
        'filename': source_path.name,
        'mime_type': mime_type,
        'size_bytes': len(file_bytes),
        'sha256_hex': file_sha,
        'source_name': 'MSO',
        'created_at': utc_now_iso(),
        'data': file_bytes,
    }

def upsert_to_db(batch_size: int = 1000) -> None:
    documents = [json.loads(line) for line in Path(OUT_DOCS_JSONL).read_text(encoding='utf-8').splitlines() if line.strip()]
    sections = [json.loads(line) for line in Path(OUT_SECTIONS_JSONL).read_text(encoding='utf-8').splitlines() if line.strip()]
    rows = [json.loads(line) for line in Path(OUT_JSONL_WITH_EMB).read_text(encoding="utf-8").splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    assert not df.empty, "Aucune ligne a inserer."
    replace_existing_docs = os.getenv("MSO_REPLACE_EXISTING_DOCS", "0").strip().lower() in {"1", "true", "yes", "y", "on"}
    df["embedding_m3"] = df[EMBED_COL].apply(vec_to_pgvector)
    df["embedding_bge_scw"] = df["embedding_bge_scw"].apply(lambda v: vec_to_pgvector(v) if isinstance(v, list) else None)
    legacy_documents = []
    for rec in documents:
        legacy_payload = make_legacy_document_payload(rec)
        if legacy_payload is not None:
            legacy_documents.append(legacy_payload)
            rec['legacy_doc_id'] = legacy_payload['id']
        else:
            rec['legacy_doc_id'] = None
        rec['source_url'] = None
        rec['metadata'] = {
            **(rec.get('metadata') or {}),
            'original_local_source_path': (rec.get('metadata') or {}).get('local_source_path'),
        }
        rec['quality_flags'] = json.dumps(rec.get('quality_flags')) if rec.get('quality_flags') is not None else None
        rec['metadata'] = json.dumps(rec.get('metadata')) if rec.get('metadata') is not None else None
        rec['doc_structure'] = json.dumps(rec.get('doc_structure')) if rec.get('doc_structure') is not None else None
    for rec in sections:
        rec['parent_section_id'] = None
        rec['section_type'] = 'heading'
        rec['metadata'] = json.dumps(rec.get('metadata') or {})
        rec['references_juridiques'] = json.dumps(rec.get('references_juridiques') or [])
    if 'references_juridiques' in df.columns:
        df['references_juridiques'] = df['references_juridiques'].apply(lambda v: json.dumps(v or []))
    if 'source_document_id' in df.columns and documents:
        doc_id = documents[0]['doc_id']
        df['source_document_id'] = df['source_document_id'].fillna(doc_id)
    with pg_conn() as con, con.cursor() as cur:
        for stmt in [s.strip() for s in CREATE_LEGACY_DOCUMENTS_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        for stmt in [s.strip() for s in CREATE_DOCUMENTS_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        for stmt in [s.strip() for s in CREATE_SECTIONS_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        for stmt in [s.strip() for s in CREATE_TABLE_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        con.commit()
        if replace_existing_docs and documents:
            doc_ids = [rec['doc_id'] for rec in documents]
            legacy_doc_ids = [rec.get('legacy_doc_id') for rec in documents if rec.get('legacy_doc_id')]
            cur.execute(f'DELETE FROM "{SCHEMA}"."{TABLE}" WHERE source_document_id = ANY(%s::uuid[])', (doc_ids,))
            cur.execute('DELETE FROM "public"."rag_sections" WHERE doc_id = ANY(%s::uuid[])', (doc_ids,))
            cur.execute('DELETE FROM "public"."rag_documents" WHERE doc_id = ANY(%s::uuid[])', (doc_ids,))
            if legacy_doc_ids:
                cur.execute('DELETE FROM "public"."documents" WHERE id = ANY(%s::uuid[])', (legacy_doc_ids,))
            con.commit()
        if legacy_documents:
            cur.executemany(UPSERT_LEGACY_DOCUMENTS_SQL, legacy_documents)
            con.commit()
        if documents:
            cur.executemany(UPSERT_DOCUMENTS_SQL, documents)
            con.commit()
        if sections:
            cur.executemany(UPSERT_SECTIONS_SQL, sections)
            con.commit()
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i + batch_size].to_dict(orient="records")
            cur.executemany(UPSERT_SQL, batch)
            con.commit()
    print(f"Upsert termine: docs={len(documents)} sections={len(sections)} chunks={len(df)} dans {SCHEMA}.{TABLE}")

## Etape 8 - Garde-fou d'execution finale

La derniere cellule ne pousse pas en base par defaut. L'upsert n'est lance que si `MSO_UPSERT_TO_DB=1`.

C'est volontaire : cela force un mode de travail prudent.

Bonne pratique de passation :

1. lancer une premiere fois sans upsert
2. verifier les `.txt`, les JSONL et les embeddings
3. seulement ensuite relancer avec l'upsert active

In [9]:
if os.getenv("MSO_UPSERT_TO_DB", "0").strip().lower() in {"1", "true", "yes", "y", "on"}:
    upsert_to_db()
else:
    print("Upsert DB ignore. Definir MSO_UPSERT_TO_DB=1 pour pousser en base.")